In [1]:
import sys
sys.path.append('../../')

from samrfi import RFIModels
import numpy as np

/home/gpuhost002/ddeal/miniconda3/envs/sam2_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from samrfi import RFIModels, RadioRFI

dir_path = '/home/gpuhost002/ddeal/RFI-AI/'

original_calib = '/home/gpuhost002/ddeal/RFI-AI/one_antenna_3C129_tfcrop.ms'
#original_calib = '/home/gpuhost002/ddeal/RFI-AI/original_calib/calib_phase_rflag.ms'

datarfi_3C129 = RadioRFI(vis=original_calib, dir_path=dir_path)
datarfi_3C129.load(mode='DATA', ant_i=1) # using first two antennas


Loading data...


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
datarfi_3C129.load(mode='FLAG', ant_i=1) # using first two antennas
datarfi_3C129.flags = np.abs(datarfi_3C129.ms_flags)

In [3]:
#datarfi_3C129.rfi_antenna_data = np.pad(datarfi_3C129.rfi_antenna_data, ((0, 0), (0, 0), (0, 0), (0, 1)), mode='wrap')
datarfi_3C129.rfi_antenna_data = datarfi_3C129.rfi_antenna_data[0:2]

In [ ]:
datarfi_3C129.rfi_antenna_data.shape

In [4]:
cd /home/gpuhost002/ddeal/RFI-AI/sam2/notebooks

/home/gpuhost002/ddeal/RFI-AI/sam2/notebooks


/home/gpuhost002/ddeal/miniconda3/envs/sam2_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
model_path = "/home/gpuhost002/ddeal/RFI-AI/samrfi_data/models/model_tfcrop_twice_patch_size-256_sam2-large_epochs60_20250110_070533.pth"
model = RFIModels(radiorfi_instance=datarfi_3C129, device='cuda',)

model.sam2_ckpt = "../checkpoints/sam2.1_hiera_large.pt"
model.sam2_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

model.device = 'cuda'

model.load_model_sam2(model_path)
model.run_model_sam2(patch_size=256, num_points=1024*4, point_threshold=.1, threshold=.9999, multimask_output=True, reuse_logits=True)

(2, 4, 2048, 511)


/home/gpuhost002/ddeal/RFI-AI/SAM-RFI/notebooks/test_notebooks/../../samrfi/rfimodels.py:109: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.sam2_predictor.model.load_st

In [ ]:
model_path = "/home/gpuhost002/ddeal/RFI-AI/samrfi_data/models/model_stretch-SQRT_sigma-2_size-256_sam2-large_epochs8_num_patches-600_20250108_181500.pth"
model = RFIModels(radiorfi_instance=datarfi_3C129, device='cuda',)

model.sam2_ckpt = "../checkpoints/sam2.1_hiera_large.pt"
model.sam2_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

model.device = 'cuda'

model.load_model_sam2(model_path)
model.run_model_sam2(patch_size=256, num_points=512, point_threshold=0.5, threshold=0.9, multimask_output=True)

In [ ]:
datarfi_3C129.radio_metrics.test_realdata()

In [ ]:
datarfi_3C129.radio_metrics.test_realdata()

In [ ]:
model.test_masks.shape

In [ ]:
model.logits.shape

In [11]:
model_thes = model.logits > 0.9

model_thes[1, 0, :, :]

tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False,  True]])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 14), dpi=200)
plt.imshow(model.logits[2, 0, :, :].T>0.5, cmap='gray',)